# `kNNpy.Auxiliary.constructingFishermatrix` Tutorial: A quick way to forecast errors from simulations by constructing a Fisher matrix from the data vectors and the fulll covariance matrix from N-body simulations.

**Author**: Anargha Mondal\
**Date**: 3 June 2026\
**kNNpy version**: 0.0.1

#### This notebook presents a simple guide on how to use the `constructingFishermatrix` function to calculate the Fisher information matrix out of given set of data vectors, a covariance matrix, and the parameters of the hyperspace. The basic functionalities of the module are illustrated using realistic cosmological N-body simulations, taken from [QUIJOTE simulations](https://quijote-simulations.readthedocs.io/en/latest/). A detailed API [documentation](https://kitnenikatnivasi.github.io/kNNpy_documentation_html/kNNpy/Auxiliary/Fisher.html) and source code for the module is available on the `kNNpy` [website](https://kitnenikatnivasi.github.io/).

## A brief introduction to Fisher matrices

Sometimes, it is useful to forecast the uncertainty of an experiment before taking any physical measurements. One way to do this is to calculate the Fisher information:
$$F_{ij}=\left\lang \frac{\partial^2\mathcal{L}}{\partial\theta_i\partial\theta_j}\right\rang$$
where $\mathcal{L}\equiv -\text{ln} L$, where $L$ is the likelihood fuction

For an $n$-dimesional parameter space, the above quantity can be represented as an $n\times n$ matrix. Fisher matrices are used frequently in the analysis of combining cosmological constraints from various data sets. They encode the Gaussian uncertainties of multiple variables.

For a Gaussian liklihood, and given cosmological parameters $\vec{{\theta}}$ and given a statistic $\vec{S}$, the Fisher matrix is defined as:
$$ F_{ij}=\displaystyle\Sigma_{\alpha,\beta}\frac{\partial S_\alpha}{\partial \theta_i}C^{-1}_{\alpha\beta}\frac{\partial S_{\beta}}{\partial \theta_j} $$
The trace term has been dropped, as it is expected to be small [Kodwani et al. 2019](https://astro.theoj.org/article/7716-the-effect-on-cosmological-parameter-estimation-of-a-parameter-dependent-covariance-matrix). 

For more information, see [Tegmark et al. 1997](https://iopscience.iop.org/article/10.1086/303939)

<!--TABLE OF CONTENTS-->
# Contents:
- [Imports and Setup](#Imports-and-Setup)
- [Data Vectors and Covariance Matrix](# Data Vectors and Covariance Matrix)
  - [Data Vectors](# Data Vectors)
  - [Computing the Covariance Matrix](# Computing the Covariance Matrix)
- [Corner Plots](#Corner Plots)
    

# Imports and Setup

#### Let's start by importing the required Python libraries. These should already be present in the `kNNpy_env` virtual environment created during the [installation](https://kitnenikatnivasi.github.io/install.html), so you should be able to import them without any issues.

In [12]:
#Importing external libraries

import numpy as np
import readgadget
import readfof

from matplotlib import pyplot as plt, ticker as mticker
import matplotlib.colors as colors

import os
import sys

import warnings
import plotting_library as PL
from pylab import *
from matplotlib.colors import LogNorm
#We prefer turning off the annoying warnings thrown by Python. Comment out the line below if you prefer to view the warnings as they arise.
warnings.filterwarnings('ignore')

import MAS_library as MASL

#Importing the kNNpy modules that will be used in this tutorial

#Necessary for relative imports (see https://stackoverflow.com/questions/34478398/import-local-function-from-a-module-housed-in-another-directory-with-relative-im)
module_path = os.path.abspath(os.path.join('../'))           # '../' is needed because the parent directory is one directories upstream of the tutorials directory
if module_path not in sys.path:
    sys.path.append(module_path)

from kNNpy.Auxiliary import Fisher as f         

#### We define our `matplotlib` plotting preferences below; you can change them to your liking. Remember to set `ustex==False` if you do not have $\LaTeX$ installed. 

In [13]:
#Matplotlib settings

plt.rcParams['font.family'] = 'serif'
plt.rc('text', usetex=True)
plt.rcParams.update({'font.size': 18})
plt.rcParams["axes.linewidth"] = 2*0.8
plt.rcParams["xtick.direction"] = 'in'
plt.rcParams["xtick.major.width"] = 2*0.8
plt.rcParams["ytick.direction"] = 'in'
plt.rcParams["ytick.major.width"] = 2*0.8
plt.rcParams["xtick.major.size"] = 2*3.5
plt.rcParams["ytick.major.size"] = 2*3.5
plt.rcParams["xtick.minor.visible"] = True
plt.rcParams["ytick.minor.visible"] = True
plt.rcParams["xtick.minor.width"] = 2*0.6
plt.rcParams["ytick.minor.width"] = 2*0.6
plt.rcParams["xtick.minor.size"] = 2*2
plt.rcParams["ytick.minor.size"] = 2*2

DefaultColorCycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
BrightColors = ['cyan', 'yellow', 'lime', 'pink', '#E0B0FF']

# Data Vectors and the Covariance Matrix

## Data Vectors

We will be comparing the Fisher constraints between kNN CDFs (calculated using [kNNpy.kNN_3D.TracerFieldCross3D](https://kitnenikatnivasi.github.io/kNNpy_documentation_html/kNNpy/kNN_3D.html#TracerFieldCross3D)) and the two-point cross correlation function (calculated using [kNNpy.Auxilliary.TPCF.TracerField3D.CrossCorr2pt](https://kitnenikatnivasi.github.io/kNNpy_documentation_html/kNNpy/Auxiliary/TPCF/3DTPCF_Tracer-Field.html#CrossCorr2pt)) for $10^5$ most massive halos cross-correlated with the total matter field. We have combined the constraints for k=1,2,3,4.

### Defining the parameters 

In [14]:

mean = np.array([0.3175, 0.6711, 0.834, 0])
num_params=4
dtheta = [0.01, 0.02, 0.01, 0.2]
custom_cet = ["#F48D69","k","#A8198F", "#90EE90"]
labels = [ r'$\Omega_m$',r'$h$', r'$\sigma_8$', r'$M_{\nu}$']  

box=1000 #Mpc/h


In [15]:
der_matrix_2PCF=np.load('/data/Data/Arka_Data/2PCFs/derivative_matrix.npy')
cov_matrix_2PCF=np.load('/data/Data/Arka_Data/2PCFs/Cov_Matrix.npy')

der_matrix_kNN=np.load('/data/Data/Arka_Data/kNN-CDFs/derivative_matrix.npy')
cov_matrix_kNN=np.load('/data/Data/Arka_Data/kNN-CDFs/Cov_Matrix.npy')
n_cov=1000

In [16]:
F_2PCF=f.constructingFishermatrix(cov_matrix_2PCF, der_matrix_2PCF, n_cov)
F_kNN=f.constructingFishermatrix(cov_matrix_kNN, der_matrix_kNN, n_cov)

ValueError: shapes (4,) and (7,7) not aligned: 4 (dim 0) != 7 (dim 0)